In [1]:
# Устанавливаем необходимые библиотеки (если их нет в Colab)
!pip install transformers torch scikit-learn pydantic -q

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
from pydantic import BaseModel, Field, ValidationError
import warnings
warnings.filterwarnings('ignore')

print("Окружение готово. GPU доступен:", torch.cuda.is_available())

Окружение готово. GPU доступен: True


In [2]:
# Pydantic-схема для фильтрации аномалий
class NomenclatureItem(BaseModel):
    raw_text: str = Field(..., min_length=5)
    target_category: str = Field(..., min_length=2)

file_path = "Выгрузка НП-ЦН 100К.csv"

# Читаем файл
df_raw = pd.read_csv(file_path, sep=';', encoding='cp1251', on_bad_lines='skip')
df_raw.columns = ['raw_text', 'target_category']
df_raw = df_raw.dropna()

valid_data = []

# Прогоняем массив через Pydantic
for idx, row in df_raw.iterrows():
    if "ВЫБЕРИТЕ КОРРЕКТНУЮ ЦН" in str(row['target_category']):
        continue
    try:
        item = NomenclatureItem(
            raw_text=str(row['raw_text']).strip().lower(),
            target_category=str(row['target_category']).strip()
        )
        valid_data.append(item.model_dump())
    except ValidationError:
        pass

df_clean = pd.DataFrame(valid_data)
print(f"Валидных строк после очистки: {len(df_clean)}")

Валидных строк после очистки: 101319


In [3]:
# Для бейзлайна отсекаем "длинный хвост" и берем Топ-50 мажоритарных классов
TOP_N_CLASSES = 50

top_classes = df_clean['target_category'].value_counts().nlargest(TOP_N_CLASSES).index
df_baseline = df_clean[df_clean['target_category'].isin(top_classes)].copy()

print(f"Строк в Baseline-выборке (Топ-{TOP_N_CLASSES} классов): {len(df_baseline)}")

# Разбиваем на Train и Test со стратификацией
X = df_baseline['raw_text']
y = df_baseline['target_category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Размер обучающей выборки: {len(X_train)} строк")
print(f"Размер тестовой выборки: {len(X_test)} строк")

Строк в Baseline-выборке (Топ-50 классов): 10298
Размер обучающей выборки: 8238 строк
Размер тестовой выборки: 2060 строк


In [4]:
print("--- Обучение простых моделей (TF-IDF) ---")

# Векторизация текста
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Модель 1: Logistic Regression
logreg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
logreg.fit(X_train_tfidf, y_train)
y_pred_logreg = logreg.predict(X_test_tfidf)

# Модель 2: Linear SVC
svc = LinearSVC(class_weight='balanced', random_state=42)
svc.fit(X_train_tfidf, y_train)
y_pred_svc = svc.predict(X_test_tfidf)

print("Простые модели успешно обучены!")

--- Обучение простых моделей (TF-IDF) ---
Простые модели успешно обучены!


In [5]:
print("--- Обучение усложненной модели (RuBERT-tiny2) ---")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")
bert_model = AutoModel.from_pretrained("cointegrated/rubert-tiny2").to(device)

def get_embeddings(text_list, batch_size=256):
    embeddings = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True, max_length=64, return_tensors='pt').to(device)
        with torch.no_grad():
            output = bert_model(**encoded)
        # Берем эмбеддинг [CLS] токена
        batch_embeddings = output.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.extend(batch_embeddings)
    return np.array(embeddings)

# Извлекаем признаки
X_train_bert = get_embeddings(X_train.tolist())
X_test_bert = get_embeddings(X_test.tolist())

# Обучаем классификатор поверх эмбеддингов
bert_classifier = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
bert_classifier.fit(X_train_bert, y_train)
y_pred_bert = bert_classifier.predict(X_test_bert)

print("Сложная модель успешно обучена!")

--- Обучение усложненной модели (RuBERT-tiny2) ---


config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Сложная модель успешно обучена!


In [6]:
def calculate_metrics(y_true, y_pred, model_name):
    return {
        'Model': model_name,
        'F1-Weighted (Primary)': f1_score(y_true, y_pred, average='weighted'),
        'Precision (Weighted)': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'Recall (Weighted)': recall_score(y_true, y_pred, average='weighted')
    }

results = [
    calculate_metrics(y_test, y_pred_logreg, 'TF-IDF + Logistic Regression'),
    calculate_metrics(y_test, y_pred_svc, 'TF-IDF + LinearSVC'),
    calculate_metrics(y_test, y_pred_bert, 'RuBERT-tiny2 + LogReg')
]

df_results = pd.DataFrame(results).set_index('Model')

print("Сводная таблица метрик качества (Baseline):")
display(df_results.round(4))

# Небольшой текстовый вывод для куратора
print("\nАнализ результатов:")
print("1. В качестве основной метрики выбран F1-Weighted, так как он учитывает дисбаланс даже в рамках Топ-50 классов.")
print("2. TF-IDF + LinearSVC показал себя отличным быстрым бейзлайном.")
print("3. Модель на базе эмбеддингов RuBERT-tiny2 демонстрирует потенциал семантического поиска. В дальнейшем эта архитектура будет дообучаться (Fine-Tuning) для достижения целевой метрики F1 >= 0.85.")

Сводная таблица метрик качества (Baseline):


,F1-Weighted (Primary),Precision (Weighted),Recall (Weighted)
Model,,,
TF-IDF + Logistic Regression,0.8195,0.8549,0.8068
TF-IDF + LinearSVC,0.8568,0.8635,0.8597
RuBERT-tiny2 + LogReg,0.7810,0.8571,0.7544



Анализ результатов:
1. В качестве основной метрики выбран F1-Weighted, так как он учитывает дисбаланс даже в рамках Топ-50 классов.
2. TF-IDF + LinearSVC показал себя отличным быстрым бейзлайном.
3. Модель на базе эмбеддингов RuBERT-tiny2 демонстрирует потенциал семантического поиска. В дальнейшем эта архитектура будет дообучаться (Fine-Tuning) для достижения целевой метрики F1 >= 0.85.
